# pop finetune -- Phase 2 A/B finetune sweeps

GPU stage: finetunes the pretrained T5 checkpoint from `colab_pretrain.ipynb` on the CodeXGLUE refinement pairs. Two sweeps share this notebook -- edit the `--config` path in the run cell and re-run it once per config: `configs/finetune_A_ep{1,3,10}.yaml` (epoch sweep, fixed seed) then `configs/finetune_B_seed{0,1,2}.yaml` (seed sweep, fixed epochs). Before any of these: re-upload the `outputs/pretrain/final/` and `outputs/tokenizer/` directories you downloaded from `colab_pretrain.ipynb` into this Colab's `repo/outputs/` (e.g. via the Colab file browser or `google.colab.files.upload`). See `docs/gpu-reproduction.md`.

### GPU check

Make sure Colab gave you a GPU runtime (Runtime > Change runtime type > GPU) before continuing.

In [ ]:
!nvidia-smi

### Install

This clones the repo at branch `main` and installs `pop` from it. **Pin the exact commit
you actually want to run before launching** -- edit the `git clone` cell below to
`git clone --depth 1 <repo> repo && cd repo && git checkout <commit-sha>`, or swap the branch
name for a tag/commit once the code is on `main`. Running against a moving branch tip means
your results may not match what you reviewed in the PR.

In [ ]:
!git clone --depth 1 -b main https://github.com/yib7/Strats-for-Bug-Fixing.git repo
%cd repo
%pip install -q -e .

### Weights & Biases login

Run the cell below and paste your own W&B API key when prompted (interactive login -- your key
is never stored in this notebook or read by anyone else). If you skip this cell, training still
runs; `pop` auto-disables W&B reporting when `WANDB_API_KEY` isn't set (see
`pop.train.pretrain`/`pop.train.finetune`).

In [ ]:
import wandb

wandb.login()

### Run one finetune config

Edit `CONFIG` below to the config you're running this pass, then execute. Repeat once per config in `configs/finetune_A_ep*.yaml` / `configs/finetune_B_seed*.yaml`.

In [ ]:
CONFIG = "configs/finetune_A_ep1.yaml"  # edit per run
!pop finetune --config {CONFIG}

### Download results

Bring the results JSON (and, for pretrain/finetune, the checkpoint directories under
`outputs/`) back to your local clone -- see `docs/gpu-reproduction.md` for exactly where each phase's
artifacts belong.

Each config writes its own `outputs/<output_dir>/best/` checkpoint (no `results/*.json` yet -- that comes from evaluating the checkpoint, next cycle's eval/generation step). Zip and download each `outputs/finetune_*/best/` directory you care about after every config's run; adjust the path in the download cell above to match.

In [ ]:
from google.colab import files

files.download("outputs/finetune_A_ep1/best")  # noqa: adjust/comment out paths you don't need